# KeetaSense - train the insect detector on Google Colab (free GPU)

Fine-tunes **YOLOv8n** on the Wageningen *Yellow Sticky Traps* dataset, evaluates it on the
**real test split**, and gives you ONE zip to download: `best.pt` + evaluation report + the exact
config used. On the laptop: `python -m ml.install_model keetasense_model_<date>.zip`.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`
(after editing the settings cell if needed).

## 0. Settings (edit here)

In [ ]:
# --- training settings -------------------------------------------------------
USE_TILES  = True    # True = train on 640 px tiles cut from the full photos (recommended: insects are tiny)
TILE_SIZE  = 640     # tile size in pixels (also the training image size)
OVERLAP    = 0.2     # overlap between neighbouring tiles
EPOCHS     = 100     # early stopping (patience 30) may stop sooner
BATCH      = 32
AUG        = "camera"  # "camera" = LED colour cast, noise, JPEG, slight perspective; "basic" = blur/brightness only
REPORT     = "real_test"   # name of the evaluation report (ml/reports/eval_<REPORT>.md)

## 1. Get the KeetaSense code
Option A: upload a zip of the project folder (without `.venv`, `.tools`, `node_modules`) in the Files panel.
Option B: clone your team's GitHub repo (edit the URL).

In [ ]:
# Option A:
# !unzip -q keetasense.zip -d /content/keetasense
# Option B:
# !git clone https://github.com/<your-team>/keetasense.git /content/keetasense
%cd /content/keetasense
!ls

In [ ]:
# Colab already has a GPU build of PyTorch, so install only the other ML packages (pinned).
!pip install -q ultralytics==8.4.187 sahi==0.12.8 albumentations==2.0.8 pyyaml==6.0.3 python-dotenv==1.2.4
import torch; print('GPU available:', torch.cuda.is_available())

## 2. Get the dataset into `ml/data/raw/`
Download `data.zip` (~800 MB, CC0) from https://data.4tu.nl/articles/_/12707066/1 and put it on your Google Drive.
`ml.download_4tu` checks its size + MD5 and unpacks it.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p ml/data/raw && cp /content/drive/MyDrive/data.zip ml/data/raw/
!python -m ml.download_4tu
# Kaggle fallback (upload kaggle.json to the project root first):
# !pip install -q kaggle==1.7.4.5 && python -m ml.download_kaggle --dataset <owner>/<name>

## 3. Convert to YOLO format (70/15/15 split) + dataset summary, then (optionally) tile

In [ ]:
!python -m ml.convert_dataset
!cat ml/reports/dataset_summary.md
if USE_TILES:
    !python -m ml.tile_dataset --data ml/data/yolo --out ml/data/yolo_tiles --tile {TILE_SIZE} --overlap {OVERLAP}
    DATA = "ml/data/yolo_tiles"
else:
    DATA = "ml/data/yolo"
print("Training on", DATA)

## 4. Train

In [ ]:
!python -m ml.train --data {DATA}/data.yaml --epochs {EPOCHS} --imgsz {TILE_SIZE} --batch {BATCH} --device 0 --aug {AUG}

## 5. Evaluate on the REAL full-size test split (YOLO + SAHI, the same pipeline the app uses)

In [ ]:
!python -m ml.evaluate --data ml/data/yolo/data.yaml --weights ml/weights/best.pt --name {REPORT}
!cat ml/reports/eval_{REPORT}.md

## 6. Package and download (one zip)

In [ ]:
import glob
!python -m ml.package_model --report {REPORT} --data {DATA}
zip_path = sorted(glob.glob('ml/exports/keetasense_model_*.zip'))[-1]
print(zip_path)
from google.colab import files
files.download(zip_path)

On the laptop:
```
python -m ml.install_model keetasense_model_<date>.zip
```
Restart the backend: its log and every dashboard screen then show **AI model: best.pt, trained <date>**,
and the **Model** page shows this evaluation report.